<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/stm_trimming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [130]:
!pip install -U langchain-openai langgraph-checkpoint-postgres psycopg2

In [131]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages.utils import trim_messages, count_tokens_approximately

In [132]:
api_key = userdata.get("OPENAI_API_KEY")
model = ChatOpenAI(api_key=api_key, model="gpt-6-luna")

In [133]:
MAX_TOKENS = 80

In [134]:
def call_model(state: MessagesState):

  # Trim conversation history -> last N messages that fit within the token limit
  messages = trim_messages(
      state["messages"],
      strategy="last",
      token_counter=count_tokens_approximately,
      max_tokens=MAX_TOKENS,
      # start_on="human",
      # include_system=True
  )

  print("Current token count: ", count_tokens_approximately(messages=messages))

  print("Messages after trimming.")
  print("-"*80)
  for message in messages:
    print(type(message).__name__, ":", message.content)
  print("-"*80)

  response = model.invoke(state["messages"])

  return {"messages": [response]}

In [135]:
graph = StateGraph(MessagesState)
graph.add_node("call_model", call_model)

graph.add_edge(START, "call_model")
graph.add_edge("call_model", END)

In [136]:
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [137]:
config1 = {"configurable": {"thread_id": "thread-1"}}
result = workflow.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]}, config=config1)

print("recent response: ",result["messages"][-1].content)

Current token count:  11
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : Hi, My name is James Bond.
--------------------------------------------------------------------------------
recent response:  Hi James Bond, nice to meet you. How can I help?


In [138]:
config1 = {"configurable": {"thread_id": "thread-1"}}
result = workflow.invoke({"messages": [{"role": "user", "content": "Am i relevant in this AI agent era?"}]}, config=config1)

print("recent response: ",result["messages"][-1].content)

Current token count:  42
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : Hi, My name is James Bond.
AIMessage : Hi James Bond, nice to meet you. How can I help?
HumanMessage : Am i relevant in this AI agent era?
--------------------------------------------------------------------------------
recent response:  Yes. But “relevant” doesn’t mean doing the same tasks AI agents can now do—it means bringing judgment, context, trust, and responsibility to the work.

A good way to stay relevant is to learn how agents can help in your field, then focus on what still needs a person: choosing the right problems, checking results, understanding people, and owning the consequences. Your experience and relationships matter more when they’re paired with new tools.

If you tell me what you do—or what you’re worried AI might change—I can give you a more specific answer.


In [139]:
result = workflow.invoke({"messages": [{"role": "user", "content": "I'm learning about LangChain and LangGraph."}]}, config=config1)
print("recent response: ",result["messages"][-1].content)

Current token count:  15
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : I'm learning about LangChain and LangGraph.
--------------------------------------------------------------------------------
recent response:  That’s a useful direction. **LangChain** helps connect models to tools, data, and other components; **LangGraph** helps you build stateful, multi-step agent workflows with control over branching, retries, and human approval.

To make the learning valuable, aim to build something end-to-end—not just follow tutorials:

1. Create a small app that solves a real problem.
2. Add tool use or retrieval from documents.
3. Use LangGraph when the workflow needs state, branching, or checkpoints.
4. Test failure cases and check the quality of the outputs.
5. Deploy it and explain what it can—and can’t—do.

Also learn the underlying ideas: Python, APIs, structured outputs, embeddings and retrieval, evaluation, and s

In [140]:
result = workflow.invoke({"messages": [{"role": "user", "content": "What are LLM memory? And explain short term memory thoroughly with example?"}]}, config=config1)
print("recent response: ",result["messages"][-1].content)

Current token count:  23
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : What are LLM memory? And explain short term memory thoroughly with example?
--------------------------------------------------------------------------------
recent response:  ## What is LLM memory?

An LLM usually **doesn’t remember past conversations by itself**. Each time you call it, your application sends it a prompt and whatever conversation or facts you want it to use.

So “memory” is generally an **application feature**: your system saves useful information and provides the relevant parts to the model later.

Two common kinds are:

- **Short-term memory:** Information relevant to the current conversation or task.
- **Long-term memory:** Information saved for use in future conversations, such as a user’s preferences or profile.

## Short-term memory, in detail

Short-term memory is the information an assistant carries forward **within a

In [141]:
result = workflow.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=config1)
print("recent response: ",result["messages"][-1].content)

Current token count:  8
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : What is my name?
--------------------------------------------------------------------------------
recent response:  Your name is James Bond.


In [142]:
graph_state = workflow.get_state(config=config1)
values = graph_state.values
for message in values.get("messages", []):
  print(type(message).__name__, ": ", message.content)

HumanMessage :  Hi, My name is James Bond.
AIMessage :  Hi James Bond, nice to meet you. How can I help?
HumanMessage :  Am i relevant in this AI agent era?
AIMessage :  Yes. But “relevant” doesn’t mean doing the same tasks AI agents can now do—it means bringing judgment, context, trust, and responsibility to the work.

A good way to stay relevant is to learn how agents can help in your field, then focus on what still needs a person: choosing the right problems, checking results, understanding people, and owning the consequences. Your experience and relationships matter more when they’re paired with new tools.

If you tell me what you do—or what you’re worried AI might change—I can give you a more specific answer.
HumanMessage :  I'm learning about LangChain and LangGraph.
AIMessage :  That’s a useful direction. **LangChain** helps connect models to tools, data, and other components; **LangGraph** helps you build stateful, multi-step agent workflows with control over branching, retries